In [ ]:
# Load the packages needed for dataset access and tabular preprocessing.
from datasets import load_dataset
import pandas as pd

c:\Users\DHARM SHAH\Desktop\Semester-3\DeepLearning\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# Load the AG News dataset, which comes with train and test splits.
raw_datasets = load_dataset("wangrongsheng/ag_news")

In [ ]:
# Convert the training split into a DataFrame so we can inspect the text and labels.
df = pd.DataFrame(raw_datasets['train'])
print(df.head())

                                                text  label
0  Wall St. Bears Claw Back Into the Black (Reute...      2
1  Carlyle Looks Toward Commercial Aerospace (Reu...      2
2  Oil and Economy Cloud Stocks' Outlook (Reuters...      2
3  Iraq Halts Oil Exports from Main Southern Pipe...      2
4  Oil prices soar to all-time record, posing new...      2


In [11]:
import re

# Import the Keras utilities needed for tokenization, padding, and one-hot labels.
from tensorflow.keras.layers import Dense, Embedding, SimpleRNN
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import to_categorical

# Work on copies so the original dataset stays unchanged.
train_df = df.copy()
test_df = pd.DataFrame(raw_datasets["test"])


def clean_text(text):
    # Lowercase the text and remove punctuation, special characters, and numbers.
    text = str(text).lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


# Clean the article text before building the vocabulary.
train_texts = train_df["text"].apply(clean_text).tolist()
test_texts = test_df["text"].apply(clean_text).tolist()
train_labels = train_df["label"].to_numpy()
test_labels = test_df["label"].to_numpy()

max_words = 20000
max_len = 50

# Build the vocabulary from the cleaned training text only.
tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(train_texts)

# Convert words to integer sequences, then pad or truncate to a fixed length.
X_train = pad_sequences(
    tokenizer.texts_to_sequences(train_texts),
    maxlen=max_len,
    padding="post",
    truncating="post",
)
X_test = pad_sequences(
    tokenizer.texts_to_sequences(test_texts),
    maxlen=max_len,
    padding="post",
    truncating="post",
)

# One-hot encode the labels so they match the softmax output layer.
num_classes = len(train_df["label"].unique())
y_train = to_categorical(train_labels, num_classes=num_classes)
y_test = to_categorical(test_labels, num_classes=num_classes)

print("Train shape:", X_train.shape, y_train.shape)
print("Test shape:", X_test.shape, y_test.shape)
print("Vocabulary size:", min(max_words, len(tokenizer.word_index) + 1))

Train shape: (120000, 50) (120000, 4)
Test shape: (7600, 50) (7600, 4)
Vocabulary size: 20000


In [ ]:
# Define a simple RNN text classifier with embeddings, recurrent memory, and a softmax output.
embedding_dim = 128

model = Sequential([
    Embedding(input_dim=min(max_words, len(tokenizer.word_index) + 1), output_dim=embedding_dim),
    SimpleRNN(64),
    Dense(num_classes, activation="softmax"),
])

# Build the network with the fixed sequence length, then compile it for multi-class classification.
model.build((None, max_len))
model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 50, 128)        │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_1 (SimpleRNN)        │ (None, 64)             │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,572,612 (9.81 MB)

 Trainable params: 2,572,612 (9.81 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
# Train the model and keep part of the training data for validation monitoring.
history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=128,
    validation_split=0.2,
    verbose=1,
)

# Evaluate the final model on the held-out test split.
test_loss, test_accuracy = model.evaluate(X_test, y_test, verbose=1)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

Epoch 1/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 26s 32ms/step - accuracy: 0.8145 - loss: 0.5113 - val_accuracy: 0.8650 - val_loss: 0.4038
Epoch 2/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 23s 31ms/step - accuracy: 0.9093 - loss: 0.2845 - val_accuracy: 0.8699 - val_loss: 0.3998
Epoch 3/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 22s 29ms/step - accuracy: 0.9314 - loss: 0.2146 - val_accuracy: 0.8737 - val_loss: 0.3826
Epoch 4/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 22s 29ms/step - accuracy: 0.9446 - loss: 0.1699 - val_accuracy: 0.8614 - val_loss: 0.4386
Epoch 5/5
750/750 ━━━━━━━━━━━━━━━━━━━━ 25s 34ms/step - accuracy: 0.9548 - loss: 0.1363 - val_accuracy: 0.8694 - val_loss: 0.4796
238/238 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8824 - loss: 0.4280
Test Loss: 0.4280
Test Accuracy: 0.8824
